In [1]:
!pip install -q transformers==4.39.3 onnx onnxruntime


# Restart Kernel 

In [2]:
import torch
import time
import os
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print("Torch:", torch.__version__)
import transformers
print("Transformers:", transformers.__version__)


Torch: 2.7.1+cu126
Transformers: 4.39.3


In [4]:
texts = [
    "KYC means Know Your Customer and verifies customer identity.",
    "A savings account allows customers to earn interest.",
    "Fixed deposits lock funds for a fixed tenure with higher interest.",
    "Debit cards can be blocked via customer care or mobile banking.",
    "Banks must follow RBI compliance and regulatory guidelines."
]


In [5]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
model.train()


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/anaconda/envs/azureml_py38/lib/python3.10/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

# Manual Fine-Tuning Loop

In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

for epoch in range(3):
    total_loss = 0.0

    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=64
        )

        outputs = model(
            input_ids=inputs["input_ids"],
            labels=inputs["input_ids"]
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1} | Loss: {total_loss:.4f}")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

Epoch 1 | Loss: 29.3514
Epoch 2 | Loss: 21.4126
Epoch 3 | Loss: 16.9091


# Saving Fine-Tuned Model

In [7]:
os.makedirs("finetuned_model", exist_ok=True)

model.save_pretrained("finetuned_model")
tokenizer.save_pretrained("finetuned_model")


('finetuned_model/tokenizer_config.json',
 'finetuned_model/special_tokens_map.json',
 'finetuned_model/vocab.json',
 'finetuned_model/merges.txt',
 'finetuned_model/added_tokens.json',
 'finetuned_model/tokenizer.json')

# Export to ONNX (FP32)

In [8]:
dummy_input = tokenizer("Test input", return_tensors="pt")

torch.onnx.export(
    model,
    (dummy_input["input_ids"],),
    "model_fp32.onnx",
    input_names=["input_ids"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "sequence"}},
    opset_version=13
)


# INT8 Quantization (ONNX Runtime)

In [9]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="model_fp32.onnx",
    model_output="model_int8.onnx",
    weight_type=QuantType.QInt8
)


# Model Size Comparison

In [10]:
def size_mb(path):
    return round(os.path.getsize(path) / (1024 * 1024), 2)

print("FP32 model size (MB):", size_mb("model_fp32.onnx"))
print("INT8 model size (MB):", size_mb("model_int8.onnx"))


FP32 model size (MB): 460.93
INT8 model size (MB): 116.44


# Latency Comparison

In [11]:
import onnxruntime as ort

def measure_latency(model_path):
    session = ort.InferenceSession(model_path)
    input_ids = np.random.randint(0, 100, (1, 32), dtype=np.int64)

    start = time.time()
    session.run(None, {"input_ids": input_ids})
    return round((time.time() - start) * 1000, 2)

print("FP32 latency (ms):", measure_latency("model_fp32.onnx"))
print("INT8 latency (ms):", measure_latency("model_int8.onnx"))


FP32 latency (ms): 88.0
INT8 latency (ms): 28.16
